# NARW Upcall Dataset — Initial Exploration

Dataset: **ICML 2013 Whale Challenge — Right Whale Redux** (Kaggle).
Goal of this notebook: get a feel for the raw data before any modeling decisions.

We look at:
1. File counts and class balance
2. Audio metadata (sample rate, duration, channels, bit depth)
3. Filename structure and timestamp distribution
4. Audio playback of positive (upcall) and negative samples
5. Waveform plots
6. Spectrogram plots — **visualization only**, not the model's preprocessing

> All preprocessing / spectrogram parameters here are picked just to *see* the data. The model's actual preprocessing pipeline is a separate decision (Naama).

In [ ]:
import os
import re
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import soundfile as sf
import librosa
import librosa.display
from IPython.display import Audio, display

# Resolve repo root + data dir. Notebook can be opened from notebooks/ or repo root.
HERE = Path.cwd()
REPO = HERE if (HERE / "data").exists() else HERE.parent
DATA = REPO / "data" / "raw"
TRAIN_DIR = DATA / "train2"
TEST_DIR = DATA / "test2"
SUB_CSV = DATA / "sampleSubmission.csv"

assert TRAIN_DIR.exists(), f"Not found: {TRAIN_DIR}"
print("Repo root:", REPO)
print("Data root:", DATA)

RNG = random.Random(0)
plt.rcParams["figure.dpi"] = 110


## 1. File counts and class balance

Labels for training clips are encoded in the filename suffix:
`..._<id>_<label>.aif` where `label ∈ {0, 1}`.

In [ ]:
PAT = re.compile(
    r"^(?P<date>\d{8})_(?P<time>\d{6})_(?P<offset>\d+s\d+ms)_"
    r"(?P<split>TRAIN\d+|Test\d+)(?:_(?P<label>\d))?\.aif$"
)

def parse_name(name: str) -> dict | None:
    m = PAT.match(name)
    return m.groupdict() if m else None

train_files = sorted(os.listdir(TRAIN_DIR))
test_files = sorted(os.listdir(TEST_DIR))
print(f"Train files: {len(train_files):,}")
print(f"Test files:  {len(test_files):,}")

train_df = pd.DataFrame([{**parse_name(n), "file": n} for n in train_files])
train_df["label"] = train_df["label"].astype(int)
train_df.head()


In [ ]:
counts = train_df["label"].value_counts().sort_index()
pos_rate = train_df["label"].mean()
print(counts.to_string())
print(f"Positive rate: {pos_rate:.2%}")
print(f"Class ratio neg:pos = {counts[0] / counts[1]:.1f}:1")

fig, ax = plt.subplots(figsize=(4.5, 3))
bars = ax.bar(["no call (0)", "upcall (1)"], counts.values, color=["#888888", "#1f77b4"])
for b, v in zip(bars, counts.values):
    ax.text(b.get_x() + b.get_width() / 2, v, f"{v:,}", ha="center", va="bottom")
ax.set_title(f"Train class balance — positive rate {pos_rate:.1%}")
ax.set_ylabel("number of clips")
plt.tight_layout()
plt.show()


## 2. Audio metadata

Sample 200 random training clips and inspect sample rate / duration / channels / subtype.
We expect these to be uniform (Kaggle dataset is curated) but verify.

In [ ]:
sample_names = RNG.sample(train_files, 200)
rows = []
for fn in sample_names:
    info = sf.info(str(TRAIN_DIR / fn))
    rows.append({
        "sample_rate": info.samplerate,
        "channels": info.channels,
        "duration_s": info.duration,
        "frames": info.frames,
        "subtype": info.subtype,
        "format": info.format,
    })
meta = pd.DataFrame(rows)
print("Numeric summary:")
print(meta.describe(include="all"))
print()
print("Unique sample rates :", sorted(meta['sample_rate'].unique()))
print("Unique channel counts:", sorted(meta['channels'].unique()))
print("Unique durations (s):", sorted(meta['duration_s'].unique()))
print("Unique subtypes      :", sorted(meta['subtype'].unique()))
print("Unique formats       :", sorted(meta['format'].unique()))


## 3. Filename structure and timestamp distribution

Filenames look like: `20090328_000000_002s3ms_TRAIN0_0.aif`
- `20090328`           → date (YYYYMMDD)
- `000000`             → time (HHMMSS) of the parent recording
- `002s3ms`            → offset within the recording (`SsMms`)
- `TRAIN0` / `Test0`   → split + intra-split index
- `_0` / `_1`          → label (train only)

Let's see how clips are distributed over time and whether positives cluster on certain days.

In [ ]:
train_df["date"] = pd.to_datetime(train_df["date"], format="%Y%m%d")
train_df["time_str"] = train_df["time"].astype(str).str.zfill(6)
train_df["hour"] = train_df["time_str"].str[:2].astype(int)

print("Date range:", train_df["date"].min().date(), "→", train_df["date"].max().date())
print("Unique dates:", train_df["date"].nunique())
print("Unique hours:", sorted(train_df["hour"].unique()))
print()
train_df.head()


In [ ]:
by_day = (
    train_df.groupby(["date", "label"])
    .size()
    .unstack(fill_value=0)
    .rename(columns={0: "no call", 1: "upcall"})
)

fig, ax = plt.subplots(figsize=(11, 3))
by_day.plot(ax=ax, color=["#888888", "#1f77b4"], lw=1)
ax.set_title("Clips per day, by class")
ax.set_ylabel("# clips")
plt.tight_layout()
plt.show()

print(f"Days with at least one upcall : {(by_day['upcall'] > 0).sum()} / {len(by_day)}")
print(f"Max upcalls in a single day   : {by_day['upcall'].max()}")


## 4. Listen to samples

5 positive clips (upcall) and 5 negative clips (no call). Use the inline players.

In [ ]:
POS = train_df[train_df["label"] == 1].sample(5, random_state=0)["file"].tolist()
NEG = train_df[train_df["label"] == 0].sample(5, random_state=0)["file"].tolist()

# Browsers refuse to decode WAVs at the native 2 kHz sample rate, so resample to 8 kHz
# *for playback only*. The audio content (< 1 kHz) is preserved losslessly.
PLAYBACK_SR = 8000

def player(path):
    y, sr = sf.read(str(path))
    y_play = librosa.resample(y.astype(np.float32), orig_sr=sr, target_sr=PLAYBACK_SR)
    return Audio(data=y_play, rate=PLAYBACK_SR)

print("=== Positive (upcall) ===")
for fn in POS:
    print(fn)
    display(player(TRAIN_DIR / fn))


In [ ]:
print("=== Negative (no call) ===")
for fn in NEG:
    print(fn)
    display(player(TRAIN_DIR / fn))


## 5. Waveforms

4 positive vs 4 negative clips, same y-axis so amplitude is comparable.

In [ ]:
def load_clip(path: Path) -> tuple[np.ndarray, int]:
    y, sr = sf.read(str(path))
    return y.astype(np.float32), sr

fig, axes = plt.subplots(4, 2, figsize=(12, 7), sharex=True, sharey=True)
for i in range(4):
    for j, (label_name, files) in enumerate([("upcall (1)", POS), ("no call (0)", NEG)]):
        y, sr = load_clip(TRAIN_DIR / files[i])
        t = np.arange(len(y)) / sr
        axes[i, j].plot(t, y, lw=0.6, color="#1f77b4" if j == 0 else "#888888")
        axes[i, j].set_title(f"{label_name}  {files[i]}", fontsize=8)
        axes[i, j].set_ylabel("amp")
axes[-1, 0].set_xlabel("time (s)")
axes[-1, 1].set_xlabel("time (s)")
plt.tight_layout()
plt.show()


## 6. Log-mel spectrograms — same params the EfficientNet baseline sees

These are the actual mel images that the production preprocessing pipeline produces
(see [`conf/preprocess/narw_mel.yaml`](../conf/preprocess/narw_mel.yaml) and the
*Preprocessing pipelines* section of `CLAUDE.md`):

- `sr = 2000` Hz (native, no resample) · `n_fft = 256` · `win_length = 256` Hann · `hop_length = 64`
- `n_mels = 64` · `f_min = 30 Hz` · `f_max = 1000 Hz`
- `power = 2.0`, then converted to dB

The downstream steps the model also applies — min-max to [0, 1] → replicate to 3 channels →
resize to 300×300 → ImageNet mean/std — are skipped here so the dB scale stays human-readable.


In [ ]:
SR = 2000
N_FFT = 256
WIN = 256
HOP = 64
N_MELS = 64
F_MIN = 30.0
F_MAX = 1000.0

fig, axes = plt.subplots(4, 2, figsize=(12, 8))
for i in range(4):
    for j, (label_name, files) in enumerate([("upcall (1)", POS), ("no call (0)", NEG)]):
        y, sr = load_clip(TRAIN_DIR / files[i])
        assert sr == SR, f"expected {SR} Hz native rate, got {sr}"
        S = librosa.feature.melspectrogram(
            y=y, sr=sr, n_fft=N_FFT, win_length=WIN, hop_length=HOP,
            n_mels=N_MELS, fmin=F_MIN, fmax=F_MAX, power=2.0,
        )
        S_db = librosa.power_to_db(S, ref=np.max, top_db=80.0)
        img = librosa.display.specshow(
            S_db, sr=sr, hop_length=HOP, fmin=F_MIN, fmax=F_MAX,
            x_axis="time", y_axis="mel", ax=axes[i, j], cmap="magma",
        )
        axes[i, j].set_title(f"{label_name}  {files[i]}", fontsize=8)
fig.colorbar(img, ax=axes, format="%+2.0f dB", shrink=0.6, label="dB rel. max")
plt.show()


## 6b. Linear-STFT spectrograms (comparison)

Same clips as above, plotted with a plain linear-frequency STFT magnitude → dB instead of
a log-mel. This is what the notebook originally showed before we locked in the log-mel
production spec — kept here so you can A/B which representation surfaces the upcall
structure more clearly.

- `sr = 2000` Hz (native) · `n_fft = 256` · `win_length = 256` Hann · `hop_length = 64`
- Linear frequency axis (capped at 500 Hz on display; NARW upcalls sit ~50–250 Hz)
- `|STFT|` → dB via `librosa.amplitude_to_db(ref=np.max, top_db=80)`


In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(12, 8))
for i in range(4):
    for j, (label_name, files) in enumerate([("upcall (1)", POS), ("no call (0)", NEG)]):
        y, sr = load_clip(TRAIN_DIR / files[i])
        S = np.abs(librosa.stft(y, n_fft=N_FFT, win_length=WIN, hop_length=HOP))
        S_db = librosa.amplitude_to_db(S, ref=np.max, top_db=80.0)
        img = librosa.display.specshow(
            S_db, sr=sr, hop_length=HOP,
            x_axis="time", y_axis="hz", ax=axes[i, j], cmap="magma",
        )
        axes[i, j].set_title(f"{label_name}  {files[i]}", fontsize=8)
        axes[i, j].set_ylim(0, 500)
fig.colorbar(img, ax=axes, format="%+2.0f dB", shrink=0.6, label="dB rel. max")
plt.show()


## Summary of raw-data facts

| | |
|---|---|
| Train clips | 47,841 |
| Test clips  | 25,468 |
| Sample rate | 2 kHz (mono, 16-bit AIFF) |
| Clip length | 2 s |
| Positive rate (train) | ~11% (~8:1 imbalance) |
| Frequency range visible | 0–1000 Hz (Nyquist) — upcalls sit ~50–250 Hz |

## Open decisions for Naama (not for Claude)

These are all separate decisions, none are made in this notebook:

- **Preprocessing / spectrogram params**: which front-end? STFT vs mel; n_fft, hop, mel bins; dB scaling; per-clip vs global normalization; input shape for ResNet-style backbone.
- **Split strategy**: random vs day-stratified vs time-based. Days/hours where positives cluster might leak across train/val if split randomly.
- **Class imbalance handling**: weighted loss vs over/under-sampling vs focal vs leave as-is.
- **Foundation model choice**: Perch (ONNX) vs BEATs.
- **Evaluation metrics**: this competition used AUC. Conservation context cares about FN cost — likely want PR-AUC and report sensitivity at a fixed FPR too. Threshold-picking strategy?
